# Basic In-Context Learning Techniques — End-to-End Pipeline
**Date**: 2026-05-31  
**Objective**: Implement and evaluate basic in-context learning techniques (zero-shot, one-shot, few-shot, structured prompting, and retrieval-augmented prompting) on a small text-classification task.

In [ ]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
import os
from pathlib import Path
import random
import re
from statistics import mean
from typing import Iterable

SEED: int = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {"0", "false", "no", "off"}

def load_runtime_env() -> dict[str, str]:
    candidate_paths = [
        Path("configs/runtime.env"),
        Path("configs/runtime.env.example"),
    ]
    values: dict[str, str] = {}
    for candidate in candidate_paths:
        if candidate.exists():
            for line in candidate.read_text(encoding="utf-8").splitlines():
                stripped = line.strip()
                if not stripped or stripped.startswith("#") or "=" not in stripped:
                    continue
                key, value = stripped.split("=", maxsplit=1)
                values[key.strip()] = value.strip()
            break
    return values

runtime_env = load_runtime_env()
use_gpu = parse_use_gpu_flag(runtime_env.get("USE_GPU", "1"))

try:
    import torch  # type: ignore

    runtime_device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
except Exception:
    runtime_device = "cpu"

print(f"SEED={SEED} | USE_GPU={int(use_gpu)} | runtime_device={runtime_device}")

In [ ]:
@dataclass(frozen=True)
class Config:
    train_fraction: float = 0.67
    max_demo_examples: int = 3

config = Config()
print(config)

In [ ]:
DATASET: list[dict[str, str]] = [
    {"text": "Payment failed after entering card details", "label": "billing"},
    {"text": "Need invoice copy for March subscription", "label": "billing"},
    {"text": "Charged twice for the same order", "label": "billing"},
    {"text": "Cannot sign in after password reset", "label": "technical"},
    {"text": "Mobile app crashes on launch", "label": "technical"},
    {"text": "Two-factor code never arrives", "label": "technical"},
    {"text": "How do I upgrade to enterprise plan", "label": "sales"},
    {"text": "Need demo for team before buying", "label": "sales"},
    {"text": "Request pricing for 500 seats", "label": "sales"},
    {"text": "Where can I download latest invoice", "label": "billing"},
    {"text": "Error 500 appears in analytics dashboard", "label": "technical"},
    {"text": "Can we get volume discount this quarter", "label": "sales"},
]

split_index = int(len(DATASET) * config.train_fraction)
train_rows = DATASET[:split_index]
test_rows = DATASET[split_index:]

print(f"train={len(train_rows)} | test={len(test_rows)}")
print("Sample train row:", train_rows[0])

In [ ]:
train_distribution = Counter(row["label"] for row in train_rows)
test_distribution = Counter(row["label"] for row in test_rows)
avg_train_tokens = mean(len(row["text"].split()) for row in train_rows)
avg_test_tokens = mean(len(row["text"].split()) for row in test_rows)

print("Train distribution:", dict(train_distribution))
print("Test distribution:", dict(test_distribution))
print(f"Avg tokens - train: {avg_train_tokens:.2f}, test: {avg_test_tokens:.2f}")

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")

def normalize_text(text: str) -> str:
    return " ".join(TOKEN_PATTERN.findall(text.lower()))

def tokenize(text: str) -> list[str]:
    return normalize_text(text).split()

for row in train_rows:
    row["clean_text"] = normalize_text(row["text"])

for row in test_rows:
    row["clean_text"] = normalize_text(row["text"])

print(train_rows[0])

In [ ]:
class ToyInContextClassifier:
    def __init__(self, labels: list[str]) -> None:
        self.labels = labels
        self.keyword_scores: dict[str, Counter[str]] = {
            label: Counter() for label in labels
        }

    def fit(self, rows: Iterable[dict[str, str]]) -> None:
        for row in rows:
            label = row["label"]
            self.keyword_scores[label].update(tokenize(row["clean_text"]))

    def _score(self, text: str) -> dict[str, int]:
        tokens = tokenize(text)
        return {
            label: sum(self.keyword_scores[label][tok] for tok in tokens)
            for label in self.labels
        }

    def zero_shot(self, text: str) -> str:
        baseline_keywords = {
            "billing": {"invoice", "charged", "payment", "card"},
            "technical": {"error", "crash", "sign", "code"},
            "sales": {"pricing", "demo", "upgrade", "discount"},
        }
        tokens = set(tokenize(text))
        scores = {label: len(tokens & words) for label, words in baseline_keywords.items()}
        return max(scores, key=scores.get)

    def one_shot(self, text: str, demo: dict[str, str]) -> str:
        predicted = self.zero_shot(text)
        demo_tokens = set(tokenize(demo["clean_text"]))
        text_tokens = set(tokenize(text))
        if len(demo_tokens & text_tokens) >= 2:
            return demo["label"]
        return predicted

    def few_shot(self, text: str, demos: list[dict[str, str]]) -> str:
        vote_counter: Counter[str] = Counter()
        text_tokens = set(tokenize(text))
        for demo in demos:
            overlap = len(set(tokenize(demo["clean_text"])) & text_tokens)
            if overlap > 0:
                vote_counter[demo["label"]] += overlap
        if vote_counter:
            return vote_counter.most_common(1)[0][0]
        return self.zero_shot(text)

    def structured_prompt(self, text: str) -> str:
        scores = self._score(text)
        return max(scores, key=scores.get)

    def rag_lite(self, text: str, knowledge_rows: list[dict[str, str]]) -> str:
        text_tokens = set(tokenize(text))
        best_label = self.zero_shot(text)
        best_overlap = 0
        for row in knowledge_rows:
            overlap = len(set(tokenize(row["clean_text"])) & text_tokens)
            if overlap > best_overlap:
                best_overlap = overlap
                best_label = row["label"]
        return best_label

labels = sorted({row["label"] for row in DATASET})
model = ToyInContextClassifier(labels=labels)

In [ ]:
model.fit(train_rows)
one_shot_demo = train_rows[0]
few_shot_demos = train_rows[: config.max_demo_examples]

print("One-shot demo:", one_shot_demo)
print("Few-shot demo count:", len(few_shot_demos))

In [ ]:
def accuracy(y_true: list[str], y_pred: list[str]) -> float:
    correct = sum(1 for truth, pred in zip(y_true, y_pred) if truth == pred)
    return correct / len(y_true)

techniques = {
    "zero_shot": lambda text: model.zero_shot(text),
    "one_shot": lambda text: model.one_shot(text, one_shot_demo),
    "few_shot": lambda text: model.few_shot(text, few_shot_demos),
    "structured": lambda text: model.structured_prompt(text),
    "rag_lite": lambda text: model.rag_lite(text, train_rows),
}

y_true = [row["label"] for row in test_rows]
results: dict[str, float] = {}
for name, predictor in techniques.items():
    y_pred = [predictor(row["clean_text"]) for row in test_rows]
    results[name] = accuracy(y_true, y_pred)

results

In [ ]:
def render_ascii_bars(scores: dict[str, float]) -> None:
    for name, value in sorted(scores.items(), key=lambda item: item[1], reverse=True):
        blocks = "#" * int(value * 20)
        print(f"{name:<12} | {blocks:<20} | {value:.2f}")

print("Technique accuracy comparison")
render_ascii_bars(results)

## Summary
- Zero-shot is the fastest baseline but is usually less robust on ambiguous text.
- One-shot improves format alignment when the example is representative.
- Few-shot and retrieval-augmented prompting generally improve stability by adding local task context.
- Structured prompting is useful for production pipelines that require deterministic output shape.
- In enterprise usage, selection quality of demonstrations and retrieved context is often a stronger lever than prompt wording alone.